# 전처리 코드

In [11]:
import joblib
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
pd.options.display.float_format = '{:,.3f}'.format
pd.set_option('display.max_columns', None)
pd.set_option('display.max_seq_items', 20)
pd.set_option('display.max_rows', 20)

import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

from sklearn.base import TransformerMixin, BaseEstimator
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import QuantileTransformer,StandardScaler

from sklearn.model_selection import train_test_split
from sklearn import set_config
set_config(transform_output="pandas")

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

import numpy_financial as npf


from sklearn.metrics import f1_score, classification_report
import xgboost as xgb

## 1. 데이터 불러오기 및 데이터 요건에 맞는 데이터 추출

- 대출요건을 충족하지 못하는 데이터 삭제

- 대출이 완료된 데이터(`Fully Paid`, `Charged Off`, `Default`)

- 개인대출(`application_type=Individual`)

**다른 test set 사용하시면 아래 셀에서 test = pd.read_csv() -> 여기에 파일명 입력하시면 됩니다.**

In [12]:
## LendingClub 데이터
train = pd.read_csv('lending_club_2020_train.csv')
test = pd.read_csv('lending_club_2020_test.csv') ## test set 새로운거 사용하시면 여기에 입력하시면 됩니다.

In [13]:
## 대출요건 충족 못하는 데이터 삭제
train = train[train['id'] != 'Loans that do not meet the credit policy']
test = test[test['id'] != 'Loans that do not meet the credit policy']

## 대출이 완료된 데이터(Fully Paid, Charged Off, Default만 추출)
train_df = train[train['loan_status'].isin(['Fully Paid', 'Charged Off', 'Default'])].copy()
test_df = test[test['loan_status'].isin(['Fully Paid', 'Charged Off', 'Default'])].copy()


## 개인대출만 추출
train_df = train_df[train_df['application_type']== 'Individual'].drop('application_type',axis=1)
test_df = test_df[test_df['application_type']== 'Individual'].drop('application_type',axis=1)

---

## 2. y 정의

- 해당 대출 IRR(`loan_IRR`)의 실제 상환 기간을 고려하여 현금흐름을 계산하고, 발행 시점의 국채수익률(`bond_IRR`) 비교하여 대출 IRR이 국채 IRR보다 크면 0, 아니면 1  

- 국채는 해당 대출 발생 시점(`issue_d`)과 대출 상환 약정 기간(`term`)을 기준으로, 3년 국채와 5년 국채 IRR 매칭

### 대출 IRR

In [14]:
## loan_IRR을 계산할 수 없는 행 제거
cols_to_check = ['issue_d', 'last_pymnt_d','loan_amnt','total_pymnt','last_pymnt_d']
train_df = train_df.dropna(subset=cols_to_check).reset_index(drop=True)
test_df  = test_df.dropna(subset=cols_to_check).reset_index(drop=True)

In [15]:
def cal_IRR(row):
    # 1) 날짜 파싱 (형식이 'Nov-2016' 이면 format="%b-%Y")
    issue = pd.to_datetime(row.get('issue_d'), format="%b-%Y", errors="coerce")
    last  = pd.to_datetime(row.get('last_pymnt_d'), format="%b-%Y", errors="coerce")
    if pd.isna(issue) or pd.isna(last):
        return np.nan

    # 2) 약정 term (예: ' 36 months' -> 36)
    try:
        term = int(str(row.get('term', '')).strip().split()[0])
    except Exception:
        return np.nan

    # 3) 실제 상환 개월수 (한 달 이내 상환도 1개월로)
    real_term = (last.year - issue.year) * 12 + (last.month - issue.month) + 1
    if pd.isna(real_term):
        return np.nan
    real_term = int(real_term)

    # 4) 연체 상환 또는 부도의 경우 약정 기간으로 계산
    if real_term > term:
        real_term = term

    # 5) 현금흐름
    try:
        loan_amnt = float(row.get('loan_amnt'))
        total_pymnt = float(row.get('total_pymnt'))
    except Exception:
        return np.nan

    if real_term < 1 or not np.isfinite(loan_amnt) or not np.isfinite(total_pymnt):
        return np.nan

    cf = [-loan_amnt] + [total_pymnt / real_term] * int(real_term)

    # 6) 월 IRR -> 연환산(%) 변환
    irr_monthly = npf.irr(cf)
    if irr_monthly is None or not np.isfinite(irr_monthly):
        return np.nan
    return ((1 + irr_monthly) ** 12 - 1) * 100


In [16]:
## IRR 계산(계산시간 좀 걸림)
train_df['REAL_IRR'] = train_df.apply(cal_IRR, axis=1)
test_df['REAL_IRR']  = test_df.apply(cal_IRR, axis=1)

In [17]:
## IRR 계산이 안되는 데이터 제거
train_df = train_df.dropna(subset=['REAL_IRR']).reset_index(drop=True)
test_df  = test_df.dropna(subset=['REAL_IRR']).reset_index(drop=True)

### 국채 IRR

In [18]:
## 국채 IRR 데이터를 lendingClub에 Join
# 미 국채 3년,5년 데이터
treasury_bond_3y = pd.read_csv("GS3.csv")  ## 3년
treasury_bond_5y = pd.read_csv("GS5.csv")  ## 5년

In [19]:
def get_rf_rate(row, tres3, tres5):
    """
    row: train_df/test_df의 행
    tres3: 3년물 국채 DataFrame (observation_date, GS3)
    tres5: 5년물 국채 DataFrame (observation_date, GS5)
    """
    date = row['issue_d']

    if row['term'].strip() == '36 months':
        val = tres3.loc[tres3['observation_date'] == date, 'GS3']
        return val.values[0] if not val.empty else np.nan
    
    elif row['term'].strip() == '60 months':
        val = tres5.loc[tres5['observation_date'] == date, 'GS5']
        return val.values[0] if not val.empty else np.nan
    
    else:
        return np.nan

In [20]:
# 적용
train_df['rf_rate'] = train_df.apply(lambda row: get_rf_rate(row, treasury_bond_3y, treasury_bond_5y), axis=1)
test_df['rf_rate']  = test_df.apply(lambda row: get_rf_rate(row, treasury_bond_3y, treasury_bond_5y), axis=1)

In [21]:
train_df['rf_rate']

0         1.070
1         1.680
2         1.480
3         1.010
4         2.190
           ... 
1072574   0.850
1072575   0.820
1072576   0.870
1072577   0.400
1072578   2.360
Name: rf_rate, Length: 1072579, dtype: float64

### y값 추가

In [22]:
train_df['y'] = (train_df['REAL_IRR'] <= train_df['rf_rate']).astype(int)
test_df['y']  = (test_df['REAL_IRR'] <= test_df['rf_rate']).astype(int)

---

## 3. 전처리

- `revol_util`에 % 제거 후 수치형 변환  

- `fico_range_high` 와 `fico_range_low`는 평균내어 `fico_avg` 열 만들고 제거  

- 사후정보와 모델링에 사용하지 않을 열 제거  

- 범주형 변수 encoding  

- 수치형 변수 결측치 보간 후 QuantileTransform 적용

In [23]:
## 개인대출 모델링 설명변수,y,사후정보 분리 -> 전처리 이후 결합
X_train = train_df.drop(['y','REAL_IRR','rf_rate'],axis=1)
y_train = train_df[['y']]
post_train = train_df[['REAL_IRR','rf_rate']]

X_test = test_df.drop(['y','REAL_IRR','rf_rate'],axis=1)
y_test = test_df[['y']]
post_test = test_df[['REAL_IRR','rf_rate']]

In [24]:
## 모델링에 사용하지 않는 열 정의
del_list = ['acc_now_delinq','collection_recovery_fee','delinq_amnt','funded_amnt','funded_amnt_inv','grade','id',
            'last_credit_pull_d','last_fico_range_high','last_fico_range_low','last_pymnt_amnt','last_pymnt_d','next_pymnt_d',
            'out_prncp','out_prncp_inv','policy_code','pymnt_plan','recoveries','title','total_pymnt_inv','total_rec_int',
            'total_rec_late_fee','total_rec_prncp','url','zip_code','hardship_flag','hardship_type','hardship_reason','hardship_status',
            'deferral_term','hardship_amount','hardship_start_date','hardship_end_date','payment_plan_start_date','hardship_length',
            'hardship_dpd','hardship_loan_status','orig_projected_additional_accrued_interest','hardship_payoff_balance_amount',
            'hardship_last_payment_amount','debt_settlement_flag','addr_state','emp_title','application_type','loan_status','earliest_cr_line',
            'num_tl_op_past_12m','open_il_12m','open_rv_12m','sub_grade','sec_app_earliest_cr_line','annual_inc_joint','dti_joint','verification_status_joint',
            'revol_bal_joint','sec_app_inq_last_6mths','sec_app_mort_acc','sec_app_open_acc','sec_app_revol_util','sec_app_open_act_il','sec_app_num_rev_accts',
            'sec_app_chargeoff_within_12_mths','sec_app_collections_12_mths_ex_med','mths_since_last_major_derog','loan_status','sec_app_fico_range_low',
            'sec_app_fico_range_high','int_rate','total_pymnt','issue_d','installment']

In [25]:
## 범주형/수치형 변수 정의
cat_col = [
    'term','emp_length', 'home_ownership',
    'verification_status', 'purpose', 'initial_list_status']


num_col=[
    'loan_amnt','annual_inc','dti','delinq_2yrs','inq_last_6mths','mths_since_last_delinq','mths_since_last_record',
    'open_acc','pub_rec','revol_bal','revol_util','total_acc','collections_12_mths_ex_med','tot_coll_amt','tot_cur_bal','open_acc_6m',
    'open_act_il','open_il_24m','mths_since_rcnt_il','total_bal_il','il_util','open_rv_24m','max_bal_bc','all_util','total_rev_hi_lim',
    'inq_fi','total_cu_tl','inq_last_12m','acc_open_past_24mths','avg_cur_bal','bc_open_to_buy','bc_util','chargeoff_within_12_mths',
    'mo_sin_old_il_acct','mo_sin_old_rev_tl_op','mo_sin_rcnt_rev_tl_op','mo_sin_rcnt_tl','mort_acc','mths_since_recent_bc','mths_since_recent_bc_dlq',
    'mths_since_recent_inq','mths_since_recent_revol_delinq','num_accts_ever_120_pd','num_actv_bc_tl','num_actv_rev_tl','num_bc_sats','num_bc_tl',
    'num_il_tl','num_op_rev_tl','num_rev_accts','num_rev_tl_bal_gt_0','num_sats','num_tl_120dpd_2m','num_tl_30dpd','num_tl_90g_dpd_24m',
    'pct_tl_nvr_dlq','percent_bc_gt_75','pub_rec_bankruptcies','tax_liens','tot_hi_cred_lim','total_bal_ex_mort','total_bc_limit','total_il_high_credit_limit','avg_fico']

In [26]:
## 결측치 보간을 사용하지 않고, 결측비율이 0.4 이하인 열들을 보간할때 사용하고 버릴 열 명시(결측 비율이 0.4 이상)
high_na_col=['mths_since_last_delinq','mths_since_last_record','open_acc_6m','open_act_il','open_il_24m',
                'mths_since_rcnt_il','total_bal_il','il_util','open_rv_24m','max_bal_bc','all_util','inq_fi',
                'total_cu_tl','inq_last_12m','mths_since_recent_bc_dlq','mths_since_recent_revol_delinq']

### 전처리 파이프라인

In [27]:
# revol_util에 % 제거해주는 역할
class RevolUtilCleaner(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        if 'revol_util' in X.columns:
            X['revol_util'] = X['revol_util'].str.rstrip('%').astype(float)
        return X
        
    def set_output(self, *, transform=None):  
        return self


# fico 점수의 상한과 하한으로 평균내고 기존의 fico_range_high와 fico_range_low 제거
class FicoAverager(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        if {'fico_range_high', 'fico_range_low'}.issubset(X.columns):
            X['avg_fico'] = X[['fico_range_high', 'fico_range_low']].mean(axis=1)
            X.drop(columns=['fico_range_high', 'fico_range_low'], inplace=True)
        return X
    
    def set_output(self, *, transform=None):  
        return self

# del_list에 지정한 열들 제거
class ColumnDropper(BaseEstimator, TransformerMixin):
    def __init__(self, drop_cols):
        self.drop_cols = drop_cols

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.drop(columns=self.drop_cols, errors='ignore')
    
    def set_output(self, *, transform=None):  
        return self

## 범주형 변수 encoder

# 1. TermEncoder -> 60개월이면 1, 36개월이면 0
class TermEncoder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        series = self._to_series(X)
        encoded = np.where(series.str.strip() == '60 months', 1, 0)
        return pd.DataFrame(encoded, columns=self.get_feature_names_out(), index=series.index)

    def get_feature_names_out(self, input_features=None):
        return ['60_months']

    def set_output(self, *, transform=None):
        return self

    def _to_series(self, X):
        return X.iloc[:, 0] if isinstance(X, pd.DataFrame) else pd.Series(X[:, 0], index=np.arange(X.shape[0]))


# 2. EmpLengthEncoder -> Unknown부터 10+ years 오름차순 순으로 encoding
class EmpLengthEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.categories_ = [
            'Unknown', '< 1 year', '1 year', '2 years', '3 years', '4 years',
            '5 years', '6 years', '7 years', '8 years', '9 years', '10+ years'
        ]
        self.category_to_int_ = {cat: idx for idx, cat in enumerate(self.categories_)}

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        series = self._to_series(X).fillna('Unknown')
        encoded = series.map(self.category_to_int_).fillna(0).astype(int)
        return pd.DataFrame(encoded.values, columns=self.get_feature_names_out(), index=series.index)

    def get_feature_names_out(self, input_features=None):
        return ['emp_length_encoded']

    def set_output(self, *, transform=None):
        return self

    def _to_series(self, X):
        return X.iloc[:, 0] if isinstance(X, pd.DataFrame) else pd.Series(X[:, 0], index=np.arange(X.shape[0]))


# 3. HomeOwnershipEncoder -> ANY, NONE, OTHER를 OTHER로 묶고, One-Hot encoding(OTHER는 encoding없음)
class HomeOwnershipEncoder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X_ = self._combine_rare_categories(X)
        dummies = pd.get_dummies(X_.iloc[:, 0], prefix='home_ownership')
        if 'home_ownership_OTHER' in dummies.columns:
            dummies.drop(columns='home_ownership_OTHER', inplace=True)
        self.ohe_columns = dummies.columns.tolist()
        return self

    def transform(self, X):
        X_ = self._combine_rare_categories(X)
        dummies = pd.get_dummies(X_.iloc[:, 0], prefix='home_ownership')
        if 'home_ownership_OTHER' in dummies.columns:
            dummies.drop(columns='home_ownership_OTHER', inplace=True)
        for col in self.ohe_columns:
            if col not in dummies.columns:
                dummies[col] = 0
        return dummies[self.ohe_columns].astype(int)

    def get_feature_names_out(self, input_features=None):
        return self.ohe_columns

    def _combine_rare_categories(self, X):
        series = X.iloc[:, 0] if isinstance(X, pd.DataFrame) else pd.Series(X[:, 0], index=np.arange(X.shape[0]))
        series = series.replace({'ANY': 'OTHER', 'NONE': 'OTHER', 'OTHER': 'OTHER'})
        return pd.DataFrame(series)

    def set_output(self, *, transform=None):
        return self


# 4. PurposeEncoder -> 대출목적을 상위항목으로 묶고 One-Hot encoding(other는 encoding없음)
class PurposeEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.ohe_columns = None
        self.mapping = {
            'debt_consolidation': 'finance', 'credit_card': 'finance',
            'vacation': 'life', 'moving': 'life', 'wedding': 'life',
            'medical': 'life', 'educational': 'life',
            'home_improvement': 'large_expense', 'car': 'large_expense',
            'major_purchase': 'large_expense', 'house': 'large_expense',
            'other': 'other', 'small_business': 'other', 'renewable_energy': 'other'
        }

    def fit(self, X, y=None):
        X_mapped = self._map_categories(X)
        dummies = pd.get_dummies(X_mapped.iloc[:, 0], prefix='purpose')
        self.ohe_columns = [col for col in dummies.columns if col != 'purpose_other']
        return self

    def transform(self, X):
        X_mapped = self._map_categories(X)
        dummies = pd.get_dummies(X_mapped.iloc[:, 0], prefix='purpose')
        if 'purpose_other' in dummies.columns:
            dummies.drop(columns='purpose_other', inplace=True)
        for col in self.ohe_columns:
            if col not in dummies.columns:
                dummies[col] = 0
        return dummies[self.ohe_columns].astype(int)

    def _map_categories(self, X):
        series = X.iloc[:, 0] if isinstance(X, pd.DataFrame) else pd.Series(X[:, 0], index=np.arange(X.shape[0]))
        mapped = series.map(self.mapping).fillna('other')
        return pd.DataFrame(mapped)

    def get_feature_names_out(self, input_features=None):
        return self.ohe_columns

    def set_output(self, *, transform=None):
        return self


# 5. VerificationStatusEncoder -> Source Verified도 Verified로 취급
class VerificationStatusEncoder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.input_features_ = list(X.columns) if isinstance(X, pd.DataFrame) else None
        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X, columns=self.input_features_, index=np.arange(X.shape[0]))
        result = X.apply(lambda col: col.isin(['Verified', 'Source Verified']).astype(int))
        result.columns = self.get_feature_names_out(X.columns)
        return result

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = getattr(self, 'input_features_', [])
        return [f'{col}' for col in input_features]

    def set_output(self, *, transform=None):
        return self


# 6. InitialListStatusEncoder -> w이면 1, 아니면 0 binary encoding
class InitialListStatusEncoder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        series = X.iloc[:, 0] if isinstance(X, pd.DataFrame) else pd.Series(X[:, 0], index=np.arange(X.shape[0]))
        encoded = np.where(series.str.strip() == 'w', 1, 0)
        return pd.DataFrame(encoded, columns=self.get_feature_names_out(), index=series.index)

    def get_feature_names_out(self, input_features=None):
        return ['w']

    def set_output(self, *, transform=None):
        return self
    

##수치형 변수 전처리 파이프라인

# 결측치 보간기
# 각 열을 보간할 때, 결측패턴 유사도가 0.5 이상인 열을 제외하고 보간을 진행
# 결측치 비율이 0.4 이상인 열은 0.4 이하인 열들을 보간할 떄 사용하고 이후 drop
class CustomIterativeImputer(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        exclude_cols=None,
        estimator=None,
        max_iter=10,
        sample_posterior=False,
        tol=1e-3,
        n_nearest_features=None,
        initial_strategy='mean',
        imputation_order='ascending',
        skip_complete=True,
        min_value=None,
        max_value=None,
        verbose=0,
        random_state=None,
        add_indicator=False,
        na_corr_thresh=0.5
    ):
        self.exclude_cols = exclude_cols if exclude_cols is not None else []
        self.estimator = estimator
        self.max_iter = max_iter
        self.sample_posterior = sample_posterior
        self.tol = tol
        self.n_nearest_features = n_nearest_features
        self.initial_strategy = initial_strategy
        self.imputation_order = imputation_order
        self.skip_complete = skip_complete
        self.min_value = min_value
        self.max_value = max_value
        self.verbose = verbose
        self.random_state = random_state
        self.add_indicator = add_indicator
        self.na_corr_thresh = na_corr_thresh

        self.imputers = dict()

    def fit(self, X, y=None):
        X = pd.DataFrame(X).copy()
        na_ind = X.isnull().astype(int)
        self.na_corr_matrix_ = na_ind.corr()
        all_cols = X.columns.tolist()

        self.valid_cols_ = [col for col in all_cols if col not in self.exclude_cols]

        for col in self.valid_cols_:
            corr_with_col = self.na_corr_matrix_[col].drop(labels=[col])
            eligible_predictors = corr_with_col[abs(corr_with_col) <= self.na_corr_thresh].index.tolist()
            eligible_predictors = [col for col in eligible_predictors if col in X.columns]

            imputer = IterativeImputer(
                estimator=self.estimator,
                max_iter=self.max_iter,
                sample_posterior=self.sample_posterior,
                tol=self.tol,
                n_nearest_features=self.n_nearest_features,
                initial_strategy=self.initial_strategy,
                imputation_order=self.imputation_order,
                skip_complete=self.skip_complete,
                min_value=self.min_value,
                max_value=self.max_value,
                verbose=self.verbose,
                random_state=self.random_state,
                add_indicator=self.add_indicator
            )

            fit_df = X[[col] + eligible_predictors]
            imputer.fit(fit_df)

            self.imputers[col] = {
                'imputer': imputer,
                'predictors': eligible_predictors
            }

        return self
    
    def transform(self, X):
        X = pd.DataFrame(X).copy()
        for col, obj in self.imputers.items():
            predictors = obj['predictors']
            imputer = obj['imputer']
        
            missing_cols = [c for c in [col] + predictors if c not in X.columns]
            if missing_cols:
                raise KeyError(f"[transform 에러] '{col}' 예측에 필요한 열이 X에 없음 → {missing_cols}")
        
            input_df = X[[col] + predictors]
            imputed = imputer.transform(input_df)
            X[col] = imputed[:, 0] if not isinstance(imputed, pd.DataFrame) else imputed.iloc[:, 0]

        X.drop(columns=self.exclude_cols, inplace=True, errors='ignore')
        return X
    
    def set_output(self, *, transform=None):
        return self

## QuantileTransformer
class QTWrapper(QuantileTransformer):
    def transform(self, X):
        X_array = super().transform(X)
        if isinstance(X, pd.DataFrame):
            return pd.DataFrame(X_array, columns=X.columns, index=X.index)
        else:
            return X_array  

    def set_output(self, *, transform=None):
        return self

In [28]:
# 파이프라인 정의
pre_common_pipeline = Pipeline([
    ('revol_util_cleaner', RevolUtilCleaner()),
    ('fico_average', FicoAverager()),
    ('drop_columns', ColumnDropper(drop_cols=del_list))
])

# 범주형 transformer
categorical_transformer = ColumnTransformer(transformers=[
    ('term', TermEncoder(), ['term']),
    ('emp_length', EmpLengthEncoder(), ['emp_length']),
    ('home', HomeOwnershipEncoder(), ['home_ownership']),
    ('purpose', PurposeEncoder(), ['purpose']),
    ('Verification', VerificationStatusEncoder(), ['verification_status']),
    ('initial', InitialListStatusEncoder(), ['initial_list_status'])
], remainder='drop')

# 수치형 Imputer & Scaler
numeric_pipeline = Pipeline([
    ('imputer',CustomIterativeImputer(
    exclude_cols=high_na_col,
    na_corr_thresh=0.5,
    initial_strategy = 'median',
    max_iter=10,
    random_state=6,
    n_nearest_features=10,
    min_value=0)),

    ('scaler', QTWrapper(output_distribution='normal', random_state=6,n_quantiles=100))
])

# ColumnTransformer: 수치형 + 범주형 결합
column_transformer = ColumnTransformer(transformers=[
    ('cat', categorical_transformer, cat_col),  # cat_col: 범주형 열 리스트
    ('num', numeric_pipeline, num_col)          # num_col: 수치형 열 리스트
], remainder='drop')

In [29]:
# 전체 파이프라인: 사전 전처리 후 column_transformer 적용
full_pipeline = Pipeline([
    ('pre_common', pre_common_pipeline),
    ('features', column_transformer)
])

In [30]:
# 파이프라인의 결과를 Pandas 데이터프레임형식으로 출력하기 위한 세팅
pre_common_pipeline.set_output(transform='pandas');
column_transformer.set_output(transform="pandas");
categorical_transformer.set_output(transform="pandas");
numeric_pipeline.set_output(transform="pandas");
full_pipeline.set_output(transform='pandas')

Pipeline(steps=[('pre_common',
                 Pipeline(steps=[('revol_util_cleaner', RevolUtilCleaner()),
                                 ('fico_average', FicoAverager()),
                                 ('drop_columns',
                                  ColumnDropper(drop_cols=['acc_now_delinq',
                                                           'collection_recovery_fee',
                                                           'delinq_amnt',
                                                           'funded_amnt',
                                                           'funded_amnt_inv',
                                                           'grade', 'id',
                                                           'last_credit_pull_d',
                                                           'last_fico_range_high',
                                                           'last_fico_range_low',
                                                           'last_pymnt_amnt',
                                                           'la...
                                                   'mths_since_last_record',
                                                   'open_acc', 'pub_rec',
                                                   'revol_bal', 'revol_util',
                                                   'total_acc',
                                                   'collections_12_mths_ex_med',
                                                   'tot_coll_amt',
                                                   'tot_cur_bal', 'open_acc_6m',
                                                   'open_act_il', 'open_il_24m',
                                                   'mths_since_rcnt_il',
                                                   'total_bal_il', 'il_util',
                                                   'open_rv_24m', 'max_bal_bc',
                                                   'all_util',
                                                   'total_rev_hi_lim', 'inq_fi',
                                                   'total_cu_tl',
                                                   'inq_last_12m',
                                                   'acc_open_past_24mths',
                                                   'avg_cur_bal', ...])]))])

In [ ]:
X_train_processed = full_pipeline.fit_transform(X_train)
X_test_processed = full_pipeline.transform(X_test)

In [25]:
X_train_processed

,cat__term__60_months,cat__emp_length__emp_length_encoded,cat__home__home_ownership_MORTGAGE,cat__home__home_ownership_OWN,cat__home__home_ownership_RENT,cat__purpose__purpose_finance,cat__purpose__purpose_large_expense,cat__purpose__purpose_life,cat__Verification__verification_status,cat__initial__w,num__loan_amnt,num__annual_inc,num__dti,num__delinq_2yrs,num__inq_last_6mths,num__open_acc,num__pub_rec,num__revol_bal,num__revol_util,num__total_acc,num__collections_12_mths_ex_med,num__tot_coll_amt,num__tot_cur_bal,num__total_rev_hi_lim,num__acc_open_past_24mths,num__avg_cur_bal,num__bc_open_to_buy,num__bc_util,num__chargeoff_within_12_mths,num__mo_sin_old_il_acct,num__mo_sin_old_rev_tl_op,num__mo_sin_rcnt_rev_tl_op,num__mo_sin_rcnt_tl,num__mort_acc,num__mths_since_recent_bc,num__mths_since_recent_inq,num__num_accts_ever_120_pd,num__num_actv_bc_tl,num__num_actv_rev_tl,num__num_bc_sats,num__num_bc_tl,num__num_il_tl,num__num_op_rev_tl,num__num_rev_accts,num__num_rev_tl_bal_gt_0,num__num_sats,num__num_tl_120dpd_2m,num__num_tl_30dpd,num__num_tl_90g_dpd_24m,num__pct_tl_nvr_dlq,num__percent_bc_gt_75,num__pub_rec_bankruptcies,num__tax_liens,num__tot_hi_cred_lim,num__total_bal_ex_mort,num__total_bc_limit,num__total_il_high_credit_limit,num__avg_fico
0,1,11,1,0,0,1,0,0,1,0,0.916,0.988,1.905,1.169,0.574,1.876,-5.199,0.469,0.169,2.253,-5.199,-5.199,2.133,0.215,1.120,1.026,0.086,0.240,-5.199,-0.063,1.059,-0.309,0.038,0.431,-0.574,-5.199,-5.199,1.691,1.276,1.144,1.276,2.323,0.928,1.009,1.305,1.876,-5.199,-5.199,-5.199,-0.309,-0.063,-5.199,-5.199,2.065,2.349,0.243,2.416,-0.782
1,0,0,1,0,0,0,1,0,0,1,0.389,-0.013,0.011,-5.199,-5.199,1.169,-5.199,-1.065,-1.896,0.431,-5.199,-5.199,0.070,1.343,0.309,-0.167,1.974,-1.714,-5.199,0.191,1.909,-0.309,0.038,0.799,-0.715,1.221,-5.199,1.097,0.322,1.144,0.431,0.051,1.400,0.651,0.349,1.194,-5.199,-5.199,-5.199,5.199,-5.199,-5.199,-5.199,0.421,-0.168,1.517,0.521,2.533
2,0,6,1,0,0,0,1,0,0,1,-1.335,-0.530,-1.322,-5.199,-5.199,-0.852,2.166,-0.520,-0.060,-1.550,-5.199,-5.199,0.634,-0.765,-0.417,1.147,-0.465,0.331,-5.199,1.863,-0.487,0.501,-0.153,-0.025,0.000,-0.051,-5.199,-0.165,-0.834,-0.545,-0.871,-1.169,-1.052,-1.194,-0.834,-0.889,-5.199,-5.199,-5.199,-1.550,-0.165,2.405,-5.199,0.516,-1.389,-0.565,-0.808,0.309
3,0,11,1,0,0,1,0,0,1,0,0.682,-0.178,1.170,-5.199,-5.199,1.305,-5.199,0.448,0.314,1.221,-5.199,-5.199,0.279,0.099,0.605,-0.015,-0.914,1.012,-5.199,0.620,0.331,0.417,0.560,-0.025,0.322,-0.191,2.050,0.782,1.276,0.295,0.799,0.908,1.248,1.400,1.305,1.305,-5.199,-5.199,-5.199,-1.564,1.367,-5.199,-5.199,0.361,0.725,-0.403,1.000,-0.545
4,1,6,1,0,0,1,0,0,1,0,0.572,0.759,-0.869,-5.199,0.574,-0.140,-5.199,1.620,0.153,0.349,-5.199,-5.199,1.360,1.532,0.309,1.463,0.983,0.208,-5.199,0.431,0.487,-0.473,-0.153,0.799,0.269,-0.051,-5.199,-0.165,-0.834,-0.101,0.431,-0.560,0.362,0.852,-0.834,-0.178,-5.199,-5.199,-5.199,5.199,0.191,-5.199,-5.199,1.309,0.224,1.644,-5.199,1.144
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1072574,1,3,1,0,0,1,0,0,1,1,1.207,-0.178,0.258,-5.199,-5.199,-0.362,-5.199,0.917,0.592,-0.651,-5.199,-5.199,0.805,0.491,-0.038,1.040,-0.555,0.903,-5.199,1.932,-0.282,-0.473,-0.153,-0.025,-0.852,1.305,-5.199,-0.165,0.322,-0.545,-0.560,-0.817,-0.127,-0.140,0.349,-0.389,-5.199,-5.199,-5.199,5.199,1.367,-5.199,-5.199,0.733,-0.114,0.165,-0.591,0.165
1072575,0,8,1,0,0,1,0,0,0,0,-1.533,-1.299,0.170,-5.199,-5.199,-0.589,-5.199,-0.151,1.627,-1.550,-5.199,-5.199,-0.093,-1.041,-0.038,0.087,-1.532,1.536,-5.199,0.667,-2.076,-0.667,-0.376,-0.025,-0.574,0.208,-5.199,-0.165,0.322,-0.545,-1.221,-1.169,-0.417,-1.194,0.349,-0.620,-5.199,-5.199,-5.199,5.199,1.367,-5.199,-5.199,-0.215,-1.195,-0.896,-0.754,-1.120
1072576,0,5,1,0,0,1,0,0,0,0,0.572,0.165,0.660,-5.199,-5.199,0.748,-5.199,0.020,-0.330,0.782,-5.199,-5.199,0.852,0.094,0.309,0.695,0.8

In [ ]:
## 샤프지수 계산에 필요한 사후 데이터와 y값 재결합
train_df_prc = pd.concat([X_train_processed,post_train,y_train],axis=1)
test_df_prc = pd.concat([X_test_processed,post_test,y_test],axis=1)

In [ ]:
#train_df_prc.to_csv('train_ind_prc.csv',index=False)
#test_df_prc.to_csv('test_ind_prc.csv',index=False)

---